# Cambio Climático — Marco Transversal: ENSO, MRV y Contabilidad de Carbono

## Por qué esta línea es diferente: es el marco que conecta todas las demás

El cambio climático es la única línea **transversal** del proyecto — no analiza un ecosistema particular sino el marco de forzamiento climático y la política de mitigación que afectan a todas las otras líneas. Comprender el ENSO (El Niño-Oscilación del Sur) es entender por qué los páramos se secan, los humedales se inundan o los ríos bajan de caudal en un año determinado. Comprender el sistema MRV colombiano es entender cómo el país cuenta sus emisiones, acredita reducciones y accede a financiamiento climático internacional.

Colombia ha establecido uno de los marcos de gobernanza climática más ambiciosos de América Latina: la **NDC de 2022** compromete una reducción del **51 % de las emisiones de GEI para 2030**, con respecto al escenario tendencial. El pilar operativo es el **Sistema MRV** (Monitoreo, Reporte y Verificación) y el **Registro Nacional de Reducción de Emisiones (RENARE)**, que controlan que los proyectos REDD+, NAMAs y MDL efectivamente sean lo que dicen ser.

En este notebook el foco estadístico es el **ONI (Oceanic Niño Index)**: el índice de ENSO que conecta todas las líneas temáticas. Analizarlo aquí permite construir la tabla de lags por línea que luego se usa en `config.ENSO_LAG_MESES`.

## La tabla ENSO_LAG_MESES — por qué cada línea tiene su propio lag

| Línea temática | Lag (meses) | Justificación |
|---|---|---|
| `paramos` | 2 | Respuesta térmica rápida, sin amortiguación de vaso |
| `humedales` | 3 | Tiempo de tránsito hidráulico río → vaso lacustre |
| `oferta_hidrica` | 3-4 | Acumulación en cuenca + tiempo de tránsito fluvial |
| `calidad_aire` | 1 | El Niño → sequía → incendios → PM2.5 casi inmediato |
| `cambio_climatico` | 0 | El ONI *es* la variable, no una covariable |

## Preguntas que este notebook responde

- ¿Ha cambiado la magnitud del ENSO con el tiempo? (Mann-Kendall sobre |ONI|)
- ¿Puede un modelo STL separar la tendencia de la variabilidad interanual en la serie ONI?
- ¿Qué modelo predice mejor el ONI futuro para usarlo como covariable en las otras líneas?

## Instituciones y marcos de referencia

| Institución | Rol principal |
|---|---|
| MADS | Administrador de RENARE, rector de política climática |
| IDEAM | INGEI (inventario GEI), SMByC, sistema MRV |
| UPME | Factores de emisión energía eléctrica (FECOC) |
| NOAA CPC | Publicación mensual del ONI — fuente primaria de `load_oni()` |
| IPCC | Metodología de tiers para inventarios nacionales |

> **Contexto de dominio completo:** [`docs/fuentes/cambio_climatico.md`](../../docs/fuentes/cambio_climatico.md)  
> **Bloque:** B — Transversal temática | **Línea:** `cambio_climatico`  
> **Variable central:** ONI (Oceanic Niño Index) — índice estándar de ENSO (NOAA CPC)  
> **Modelos sugeridos:** SARIMA, SARIMAX, PROPHET, XGBOOST  
> **Análisis especial:** STL para descomposición tendencia/variabilidad + Mann-Kendall sobre |ONI|

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "cambio_climatico"
VARIABLE = "temperatura"
UNIDAD = "°C"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `cambio_climatico` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "cambio_climatico.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

En este notebook, la **variable central** es el **ONI (Oceanic Niño Index)** publicado mensualmente por la NOAA Climate Prediction Center (CPC). El ONI es la anomalía de temperatura superficial del mar (SST) en la región Niño 3.4 del Pacífico central (5°N–5°S, 170°W–120°W), calculada como promedio móvil de 3 meses.

**Clasificación ENSO según ONI (config.ENSO_THRESHOLDS):**

| Categoría | Umbral ONI | Definición | Duración mínima |
|---|---|---|---|
| El Niño moderado | ≥ +0.5 °C | SST por encima de lo normal | 5 meses consecutivos |
| El Niño fuerte | ≥ +1.5 °C | Impacto regional significativo | 5 meses consecutivos |
| El Niño muy fuerte | ≥ +2.0 °C | 1997-98, 2015-16 — eventos históricos | 5 meses consecutivos |
| Neutral | -0.5 a +0.5 °C | Condiciones normales del Pacífico | — |
| La Niña moderada | ≤ -0.5 °C | SST por debajo de lo normal | 5 meses consecutivos |
| La Niña fuerte | ≤ -1.5 °C | Inundaciones severas en Colombia | 5 meses consecutivos |

`load_oni()` descarga automáticamente la serie histórica del NOAA CPC y la devuelve en formato `DataFrame` con columnas `fecha` y `oni`. La función cachea los datos localmente para evitar descargas repetidas.

**Si se analiza una variable climática colombiana (temperatura, precipitación) en lugar del ONI directamente:** El notebook también es válido para series de temperatura media nacional o anomalías de precipitación. En ese caso, el ONI se usa como covariable SARIMAX con el lag correspondiente a la línea de interés.

> **Ruta esperada para datos reales (si no se usa ONI):** `data/raw/cambio_climatico.csv`  
> Columnas mínimas: `fecha` (datetime mensual), `temperatura` (°C) o `anomalia_precipitacion` (mm).  
> Fuente alternativa: Copernicus Climate Data Store (C3S) — `ERA5-Land` para variables de superficie.

In [ ]:
# df = load_csv("data/raw/cambio_climatico.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "temperatura": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 1b. Contabilidad de Carbono — Inventario GEI por sector

Mientras el ONI captura el forzante climático externo, la **contabilidad de carbono** cuantifica
las emisiones de GEI internas del país. En Colombia el sector **AFOLU** representa ~60% de las
emisiones brutas — principalmente por deforestación en la Amazonia y los Andes.

| Registro | Gestiona | Referencia legal |
|---|---|---|
| **RENARE** | Reducciones de GEI (REDD+, MDL, NAMAs) | Ley 1753/2015, Res. 1447/2018 |
| **INGEI** | Inventario Nacional de GEI | IDEAM / Directrices IPCC |
| **SMByC** | Monitoreo de Bosques y Carbono | IDEAM — Satelital (Landsat/Sentinel) |

> **Meta NDC Colombia:** reducir 51% de emisiones netas para 2030 (tope: 169.440 kt CO2eq).

**Formula IPCC (Tier 1):**
```
Emisiones (tCO2e) = Datos_Actividad (AD) x Factor_Emision (EF) x PCG
Ej: ha_deforestadas x EF_biomasa_tC_ha x (44/12)  ->  tCO2e liberadas
```

In [ ]:
# Inventario GEI sintetico por sector (kt CO2e/ano)
# Estructura basada en INGEI Colombia 2015-2024
sectores = ['AFOLU', 'Energia', 'Transporte', 'Industria', 'Residuos']
n_anos = 10
anos = list(range(2015, 2015 + n_anos))

np.random.seed(99)
emisiones_kt = {
    'AFOLU':      [210 - i*2   + np.random.normal(0, 5)   for i in range(n_anos)],
    'Energia':    [65  + i*0.5 + np.random.normal(0, 2)   for i in range(n_anos)],
    'Transporte': [40  + i*0.8 + np.random.normal(0, 1.5) for i in range(n_anos)],
    'Industria':  [20  - i*0.1 + np.random.normal(0, 1)   for i in range(n_anos)],
    'Residuos':   [8   + i*0.2 + np.random.normal(0, 0.5) for i in range(n_anos)],
}
df_gei = pd.DataFrame({'ano': anos,
                        **{s: [round(v, 1) for v in vals]
                           for s, vals in emisiones_kt.items()}})
df_gei['total_kt'] = df_gei[sectores].sum(axis=1).round(1)

NDC_META_KT   = 169.4
NDC_BASE_2015 = float(df_gei.loc[df_gei['ano']==2015, 'total_kt'].values[0])

print(f'Linea base 2015: {NDC_BASE_2015:.1f} kt CO2e')
print(f'Meta NDC 2030:   {NDC_META_KT:.1f} kt CO2e (reduccion {(1-NDC_META_KT/NDC_BASE_2015)*100:.0f}%)')
df_gei

## 2. Validación y EDA

La serie ONI es excepcionalmente limpia: viene de NOAA sin faltantes desde 1950. Sin embargo, si se trabaja con datos climáticos colombianos de estaciones IDEAM, aplican las mismas consideraciones de calidad que en otras líneas.

**Por qué `validate()` con `linea_tematica="cambio_climatico"` (ADR-006):** El validador aplica rangos físicos para el ONI (-3.5 a +3.5 °C) y para variables climáticas de referencia. Un ONI de +4.5 °C nunca ha ocurrido en el registro instrumental — sería un error de datos. El El Niño de 1997-1998 alcanzó +2.4 °C en su pico máximo.

**Qué buscar en el EDA del ONI:**

1. **Espectro de frecuencias:** El ENSO tiene periodicidad de **2 a 7 años**. El análisis espectral (FFT o periodograma) debe mostrar energía concentrada en ese rango. Si la serie es muy corta (< 30 años), la estimación espectral será ruidosa.

2. **Frecuencia de eventos extremos:** ¿Hay más eventos de El Niño fuerte (> 1.5 °C) en la segunda mitad del registro? Esto es evidencia del posible intensificación del ENSO con el cambio climático — la hipótesis central de Mann-Kendall sobre |ONI|.

3. **Asimetría temporal:** Los episodios de El Niño tienden a ser más cortos e intensos que los de La Niña. Esta asimetría tiene implicaciones directas para el agua en Colombia: las sequías son cortas pero severas; los períodos húmedos son más prolongados pero más moderados.

**Riesgo de datos:** Si se usan datos de temperatura de estaciones colombianas, el sesgo principal es la **baja densidad de estaciones en la Amazonia y el Pacífico**, que son las regiones más afectadas por el ENSO.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_cambio_climatico.html",
       title="EDA — Cambio Climático", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria — Interpretando el ONI en contexto colombiano

La visualización del ONI debe contextualizarse con los eventos históricos que Colombia ha experimentado:

**Eventos ENSO relevantes para Colombia (1980-2026):**

| Período | Tipo | ONI máximo | Impacto en Colombia |
|---|---|---|---|
| 1982-1983 | El Niño fuerte | +2.1 °C | Sequía severa, reducción de caudales 40-60 % |
| 1997-1998 | El Niño muy fuerte | +2.4 °C | El más intenso del s. XX; incendios en Amazonia, crisis hídrica |
| 1999-2001 | La Niña fuerte | -1.7 °C | Inundaciones en cuencas Magdalena y Cauca |
| 2007-2008 | La Niña moderada | -1.2 °C | Aumento de precipitaciones, alertas de inundación |
| 2010-2011 | La Niña fuerte | -1.5 °C | La mayor emergencia hídrica de Colombia en décadas |
| 2015-2016 | El Niño muy fuerte | +2.3 °C | Segundo más intenso del s. XX; sequía extrema en páramos |
| 2020-2021 | La Niña doble dip | -1.3 °C | Lluvias intensas, deslizamientos |

La gráfica de la serie debe mostrar claramente estos eventos. Agregar líneas horizontales en ONI = +0.5 y -0.5 (umbrales de ENSO) y en ONI = +1.5 y -1.5 (eventos fuertes).

`plot_seasonal_means` aplicado al ONI mensual muestra cuándo tienden a desarrollarse y madurar los eventos ENSO. El Niño típicamente madura en diciembre-febrero; La Niña en octubre-diciembre.

In [ ]:
plot_series(df, "fecha", "temperatura", title="Cambio Climático — temperatura (°C)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "temperatura", period="month")
plt.show()

## 3c. Visualizacion NDC — Progreso y emisiones por sector

Los dos paneles responden las preguntas clave del ciclo MRV:
- **Izquierda:** evolucion de emisiones por sector — que sector lidera la reduccion?
- **Derecha:** a que distancia esta Colombia de la meta NDC del 51% para 2030?

> AFOLU (deforestacion) es la palanca principal: una reduccion del 30% en deforestacion
> equivale a ~60 kt CO2e menos — mas que todos los demas sectores combinados.

In [ ]:
colors_sect = ['#27ae60','#e74c3c','#3498db','#f39c12','#9b59b6']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: emisiones stacked por sector
bottom = np.zeros(n_anos)
for sector, color in zip(sectores, colors_sect):
    ax1.bar(df_gei['ano'], df_gei[sector], bottom=bottom,
            label=sector, color=color, alpha=0.85, width=0.7)
    bottom += df_gei[sector].values
ax1.axhline(NDC_META_KT, color='black', ls='--', lw=2,
            label=f'Meta NDC 2030 ({NDC_META_KT} kt CO2e)')
ax1.set_title('Emisiones GEI por sector (kt CO2e/ano)', fontweight='bold')
ax1.set_ylabel('kt CO2e')
ax1.legend(fontsize=7, loc='upper right'); ax1.grid(axis='y', alpha=0.3)

# Panel B: progreso NDC (%)
pct_red = ((NDC_BASE_2015 - df_gei['total_kt']) / NDC_BASE_2015 * 100).clip(lower=0)
ax2.plot(df_gei['ano'], pct_red, marker='o', color='#27ae60', lw=2, label='Reduccion observada')
ax2.axhline(51, color='red', ls='--', lw=1.5, label='Meta NDC 51%')
ax2.fill_between(df_gei['ano'], pct_red, 51,
                 where=(pct_red < 51), alpha=0.15, color='red', label='Brecha pendiente')
ax2.set_title('Progreso NDC — % reduccion vs. linea base 2015', fontweight='bold')
ax2.set_ylabel('% reduccion'); ax2.set_ylim(0, 60)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Reduccion {anos[-1]}: {pct_red.iloc[-1]:.1f}% | Brecha meta 51%: {51 - pct_red.iloc[-1]:.1f}pp')

## 3b. El ONI como variable central — STL y descomposición de señales

En esta línea el ONI no es una covariable sino la **variable de análisis principal**. El objetivo es separar tres componentes:

1. **Tendencia de largo plazo:** ¿Está cambiando la intensidad media del ENSO? La hipótesis del cambio climático postula que los eventos extremos se intensifican. Una tendencia positiva en |ONI| (valor absoluto) sería evidencia en esta dirección.

2. **Variabilidad interanual (ciclo ENSO):** El ciclo de 2-7 años es el componente más importante operativamente — es lo que permite construir pronósticos de 6-12 meses para alertas tempranas en Colombia.

3. **Componente residual:** ruido no explicado por tendencia ni ciclo ENSO. En el ONI este componente es mínimo (la serie es suave por diseño del promedio móvil de 3 meses).

**STL (Seasonal-Trend decomposition using LOESS):** aplicado al ONI con período 7 años (84 meses, período medio del ciclo ENSO) separa estas tres señales. El resultado permite verificar si la tendencia extraída es estadísticamente significativa o artefacto del período analizado.

**Nota ADR-007:** Los lags en `config.ENSO_LAG_MESES` se definieron a partir de análisis de correlación cruzada entre el ONI y las variables de cada línea temática. Si los datos de producción sugieren un lag diferente al configurado, documentar la evidencia en `docs/decisiones.md` y actualizar `config.py`. No modificar el lag en el código del notebook sin actualizar la config central.

In [ ]:
# --- Covariable ENSO (lag específico para Cambio Climático) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="cambio_climatico")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva — El ONI en números

Los estadísticos descriptivos del ONI tienen interpretaciones climáticas directas:

- **Media:** debe ser cercana a 0 en el largo plazo (por definición de anomalía). Si la media del período analizado es positiva, el período tuvo más meses de El Niño que de La Niña — como ocurrió en las décadas de 1980-1990.
- **Desviación estándar:** típicamente entre 0.7 y 0.9 °C para la serie histórica completa. Una SD mayor en la segunda mitad del registro sería evidencia de intensificación del ENSO.
- **Percentil 5 y 95:** delimitan los eventos extremos. P95 ≈ +1.2 a +1.5 °C; P5 ≈ -1.0 a -1.3 °C. Usar estos percentiles para clasificar años en: Niño fuerte, Niño moderado, neutral, Niña moderada, Niña fuerte.
- **Frecuencia de meses con |ONI| > 1.5:** este es el estadístico más relevante para la hipótesis de intensificación. Si aumenta de ~8 % antes de 1980 a ~15 % después de 2000, hay evidencia empírica de intensificación — confirmar con Mann-Kendall.

**Para la contabilidad de carbono (variables de emisión):** Si el análisis incluye la densidad de biomasa o factores de emisión, la media y SD son críticas para el análisis de incertidumbre por simulación de Monte Carlo (metodología IPCC Tier 2-3).

In [ ]:
summarize(df)

## 5. Análisis inferencial — Estacionariedad y Mann-Kendall sobre |ONI|

### 5a. Estacionariedad del ONI (ADR-004)

El ONI como serie mensual es **estacionaria en media** pero **no en varianza** en presencia de intensificación del ENSO. El par ADF + KPSS debería confirmar estacionariedad en la serie cruda (sin tendencia en el nivel), pero si se trabaja con la serie |ONI|, puede aparecer tendencia positiva.

**Interpretación esperada:**
- ADF rechaza H₀ (p < 0.05): la serie es estacionaria — los modelos ARIMA con d=0 son apropiados
- KPSS: si rechaza H₀ para la serie |ONI|, hay tendencia positiva en la magnitud de los eventos

Para modelos predictivos del ONI, no es necesario diferenciar (d=0). El ciclo ENSO es estacionario por naturaleza física — no tiene raíz unitaria en el sentido económico.

### 5b. Mann-Kendall sobre |ONI| — la pregunta de intensificación del ENSO

**La hipótesis clave de esta línea:** ¿Se están intensificando los eventos ENSO con el cambio climático? Para responderla estadísticamente, aplicar Mann-Kendall no al ONI crudo (que oscila entre positivo y negativo) sino a su **valor absoluto |ONI|**. Si la magnitud de los eventos está aumentando, |ONI| tendrá tendencia positiva.

**Interpretación de los resultados:**
- `trend = "increasing"` sobre |ONI| con `p < 0.05`: evidencia estadística de intensificación del ENSO — hallazgo con implicaciones directas para la planificación de gestión de riesgo en Colombia y para la calibración de los lags en `config.ENSO_LAG_MESES`.
- `slope` de Sen sobre |ONI|: la tasa de incremento de magnitud en °C/mes. Multiplicar por 120 para obtener el cambio en una década.
- Esta evidencia debe registrarse en `docs/decisiones.md` y comunicarse al equipo de la línea de gestión de riesgo (si existe).

**Nota metodológica:** Mann-Kendall sobre |ONI| asume independencia entre observaciones. El ONI mensual tiene alta autocorrelación (r~0.9 entre meses consecutivos). Usar la versión modificada de Mann-Kendall que corrige por autocorrelación (`pymannkendall` función `original_test` con corrección). Documentar si se corrigió o no.

In [ ]:
ts = df.set_index("fecha")["temperatura"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Marco normativo colombiano de cambio climático

Para el ONI en sí, no aplican normas de excedencia. Pero el contexto de cambio climático en Colombia está enmarcado por un conjunto normativo que da sentido a todo el análisis:

**Marco normativo clave (Colombia):**

| Instrumento | Contenido relevante | Año |
|---|---|---|
| Ley 1931 de 2018 | Directrices para gestión del cambio climático, creación del SNICC | 2018 |
| Ley 1753 de 2015 (Art. 175) | Crea el RENARE — Registro Nacional de Reducción de Emisiones | 2015 |
| Resolución 1447 de 2018 (MADS) | Reglamenta el sistema MRV y las reglas de no doble contabilidad | 2018 |
| Decreto 926 de 2017 | Impuesto al carbono — mecanismo económico de la NDC | 2017 |
| Resolución 418 de 2024 (MADS) | Transfiere RENARE del IDEAM al MADS | 2024 |

**NDC Colombia (2022) — metas cuantificadas:**
- Reducir **51 % de emisiones GEI** al 2030 vs. escenario tendencial
- Reducir deforestación al mínimo posible (hacia cero en APs — CONPES 4050)
- 169.440 kt CO₂eq es el **techo máximo** de emisiones proyectado a 2030

**Umbrales del sistema ENSO para alertas en Colombia (config.ENSO_THRESHOLDS):**
El IDEAM emite alertas hidrometeorológicas cuando el ONI supera los umbrales de La Niña fuerte (< -1.5 °C) o El Niño fuerte (> +1.5 °C). Estas alertas activan el **SNGRD** (Sistema Nacional de Gestión del Riesgo de Desastres) y son el insumo directo para los PMA de humedales y los planes de contingencia de cuencas.

In [ ]:
rep = exceedance_report(df["temperatura"], variable="temperatura")
if rep.empty:
    print("Sin normas colombianas registradas para 'temperatura'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento — La serie ONI no necesita imputación

La serie ONI del NOAA CPC no tiene faltantes en el registro moderno (desde 1950). Este bloque de preprocesamiento aplica principalmente cuando se trabaja con **datos de temperatura colombiana** de estaciones IDEAM como variable principal en lugar del ONI.

**Para datos de temperatura colombiana — criterios de imputación:**

- Brechas cortas (1-3 meses): interpolación lineal — apropiada si la brecha no cae en un evento ENSO activo
- Brechas medianas (4-6 meses): usar datos de re-análisis climático (ERA5-Land de Copernicus) como referencia para imputar estaciones con poca cobertura
- Brechas largas (> 6 meses): no imputar — documentar la limitación

**ADR-002 en contexto de cambio climático:** Los valores extremos del ONI (eventos > 2 °C como 1997-98 y 2015-16) son exactamente la señal de interés para la hipótesis de intensificación. Cualquier suavizado o truncamiento de estos valores destruye la evidencia analítica. El único tratamiento permitido es el filtrado claramente documentado con justificación física.

**Para variables de emisión (si el análisis extiende a contabilidad de carbono):** Las simulaciones de Monte Carlo para incertidumbre de factores de emisión (metodología IPCC Tier 2) requieren series completas y sin truncamiento. Ver `docs/fuentes/cambio_climatico.md` sección "Métodos estadísticos sugeridos".

In [ ]:
df_clean = impute(df.copy(), cols=["temperatura"], method="linear")
print(f"Faltantes antes: {df["temperatura"].isna().sum()} | "
      f"después: {df_clean["temperatura"].isna().sum()}")

## 7. Modelos predictivos — Forecasting del ONI y variables climáticas

### El valor predictivo del ONI para la gestión colombiana

Un pronóstico del ONI a 3-6 meses adelante tiene valor económico directo para Colombia: permite activar con anticipación los planes de contingencia de sequía o inundación, ajustar los despachos hidroeléctricos (el sistema energético colombiano depende en un 70 % de la hidroenergía), y preparar los sistemas de alerta temprana del SNGRD.

### Por qué cuatro modelos en esta línea

**SARIMA:** baseline obligatorio. El ONI tiene autocorrelación significativa y un espectro de 2-7 años que SARIMA puede capturar parcialmente con órdenes AR altos. Orden típico sugerido: (2,0,1)(0,0,0,0) — sin diferenciación ni componente estacional fijo.

**SARIMAX:** permite incorporar **la velocidad de cambio del ONI** como regresor exógeno — si el ONI está subiendo rápidamente, tiende a seguir subiendo (momentum). También puede incorporar el índice SOI (Southern Oscillation Index) como covariable complementaria al ONI.

**Prophet:** maneja bien el ciclo ENSO de múltiple periodicidad (2-7 años). Configurar `seasonality_mode="additive"` y agregar estacionalidades personalizadas de 24, 36, 48 y 84 meses (representando el rango de periodicidades ENSO).

**XGBoost con lags:** captura relaciones no lineales entre el ONI actual y el futuro. Los lags más informativos para predecir ENSO son típicamente 1, 3, 6 y 12 meses. Es el más eficaz para horizontes cortos (1-3 meses).

### Métricas de evaluación para ONI

Usar **MAE y RMSE** en °C. Un MAE de 0.3 °C es razonablemente bueno — equivale a la mitad del umbral de clasificación Niño/Niña (0.5 °C). Un modelo con MAE > 0.6 °C en los meses 4-6 de pronóstico no supera el desempeño de la climatología y no debería usarse para alertas. **No usar RMSLE** (el ONI es una anomalía que puede ser negativa — ADR-003).

In [ ]:
ts = df_clean.set_index("fecha")["temperatura"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Incertidumbre en factores de emision — Monte Carlo (Tier 1 vs. Tier 2)

Una de las principales criticas a los creditos REDD+ es el **over-crediting**: usar factores de
emision genericos (Tier 1, IPCC por defecto) en lugar de factores nacionales calibrados (Tier 2).
Monte Carlo cuantifica esa diferencia en MtCO2e con distribucion de probabilidad completa:

```
Emisiones = Area_deforestada x EF ~ Normal(mu, sigma)
n = 10.000 simulaciones
```

**Resultado clave:** Tier 2 reduce la incertidumbre porque usa datos del IFN
(Inventario Forestal Nacional) y parcelas del SMByC en lugar de promedios globales tropicales.
La diferencia de medias es el **over-crediting potencial** evitable con datos colombianos.

In [ ]:
np.random.seed(42)
N_SIM = 10_000

# Factor de emision AFOLU: biomasa aerea + subterranea (tCO2e/ha)
# Tier 1 (IPCC Tabla 4.7, bosque humedo tropical): mayor incertidumbre
# Tier 2 (SMByC/IFN Colombia): mas preciso, parcelas nacionales
EF_T1 = np.random.normal(120, 30, N_SIM)   # tCO2e/ha  +-25%
EF_T2 = np.random.normal(95,  12, N_SIM)   # tCO2e/ha  +-12%

AREA_HA = 50_000  # ha/ano de deforestacion
em_T1 = (AREA_HA * EF_T1 / 1e6).clip(min=0)  # MtCO2e
em_T2 = (AREA_HA * EF_T2 / 1e6).clip(min=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(em_T1, bins=80, alpha=0.6, color='#e74c3c',
        label=f'Tier 1 (IPCC defecto) | media={em_T1.mean():.2f}, sigma={em_T1.std():.2f} MtCO2e')
ax.hist(em_T2, bins=80, alpha=0.6, color='#27ae60',
        label=f'Tier 2 (Colombia IFN) | media={em_T2.mean():.2f}, sigma={em_T2.std():.2f} MtCO2e')
ax.axvline(em_T1.mean(), color='#e74c3c', lw=2, ls='--')
ax.axvline(em_T2.mean(), color='#27ae60', lw=2, ls='--')
ax.set_title(f'Monte Carlo — Incertidumbre AFOLU (n={N_SIM:,} sim, area={AREA_HA:,} ha)',
             fontweight='bold')
ax.set_xlabel('MtCO2e/ano'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

ci95_T1 = np.percentile(em_T1, [2.5, 97.5])
ci95_T2 = np.percentile(em_T2, [2.5, 97.5])
print(f'Tier 1 IC95%: [{ci95_T1[0]:.2f}, {ci95_T1[1]:.2f}] — rango={ci95_T1[1]-ci95_T1[0]:.2f} MtCO2e')
print(f'Tier 2 IC95%: [{ci95_T2[0]:.2f}, {ci95_T2[1]:.2f}] — rango={ci95_T2[1]-ci95_T2[0]:.2f} MtCO2e')
print(f'Reduccion de incertidumbre con Tier 2: {(1 - em_T2.std()/em_T1.std())*100:.0f}%')
print(f'Over-crediting potencial (Tier1-Tier2): {(em_T1.mean()-em_T2.mean()):.2f} MtCO2e')

## 8. Conclusiones

Al completar el análisis, documentar aquí:

- **Intensificación ENSO (Mann-Kendall sobre |ONI|):** `trend`, `p-value`, `slope |ONI| (°C/década)`
- **Componente de tendencia STL:** ¿existe drift positivo en la tendencia extraída? ¿En qué período inicia?
- **Modelo ganador para pronóstico ONI:** nombre, MAE (°C), RMSE (°C) en walk-forward validation
- **Evento ENSO más reciente documentado:** tipo, duración, ONI máximo, impacto estimado en líneas dependientes
- **Lags validados:** confirmar o proponer ajuste a `config.ENSO_LAG_MESES` con base en correlaciones cruzadas observadas

### Normativa y marco de referencia clave

| Instrumento | Aplicación |
|---|---|
| Ley 1931 de 2018 | Gestión del cambio climático en Colombia — marco general |
| Resolución 1447 de 2018 (MADS) | Sistema MRV y RENARE — contabilidad de reducciones de GEI |
| NDC Colombia 2022 | Meta: -51 % emisiones GEI al 2030 |
| `config.ENSO_THRESHOLDS` | Umbrales ONI para clasificación de eventos |
| `config.ENSO_LAG_MESES` | Lags por línea temática — calibrar con correlaciones cruzadas |

---

## 9. Cómo adaptar a datos reales

**Caso A — Análisis del ONI directamente:**

La serie ONI ya la descarga `load_oni()` de forma automática:
```python
from estadistica_ambiental.features.climate import load_oni
oni = load_oni()  # DataFrame con columnas: fecha, oni
```
No se necesita ningún archivo local. La función cachea los datos. Para actualizar al último mes publicado por NOAA, eliminar el cache en `data/raw/oni_cache.csv`.

**Caso B — Variable climática colombiana como serie principal:**

```
data/raw/cambio_climatico.csv
```
Columnas mínimas:
```
fecha,temperatura
2000-01-31,24.3
2000-02-29,24.8
...
```
Fuentes recomendadas:
- IDEAM DHIME: temperatura media de estaciones sinópticas (código de categoría `TMED`)
- Copernicus ERA5-Land: re-análisis a resolución 0.1° — descarga vía `cdsapi`
- Anomalías IDEAM: el IDEAM publica boletines con anomalías nacionales en su portal de cambio climático

**Caso C — Variables de contabilidad de carbono (extensión avanzada):**

Para análisis MRV, los datos de entrada son:
- Densidad de biomasa: SMByC del IDEAM (`data/raw/biomasa_smbyc.csv`)
- Factores de emisión de energía: UPME / FECOC (`data/raw/fecoc_upme.csv`)
- Datos RENARE: solicitar via MADS — restringidos pero disponibles para investigación

**Verificar la tabla de lags al usar datos reales:**
```python
from estadistica_ambiental.config import ENSO_LAG_MESES
print(ENSO_LAG_MESES)
# Si los análisis de correlación cruzada sugieren lags diferentes, actualizar config.py
# y documentar en docs/decisiones.md (ADR-007)
```